# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent, hand-written baseline rule that every future ML model must beat.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline Rule (in plain words):** A page is worth reviewing if it had significant traffic (>100 impressions in the prior 30-day window), AND it is either a young page (<180 days old) OR it faces high keyword competition (>0.71). The flagged pages are then sorted by their prior impressions so we review the highest-impact pages first.

**As a formula:** `score = max(is_young, is_high_comp) × impressions_prev_30d`

**Reason Codes:**
- `YOUNG_AND_HIGH_COMP`: The page is less than 6 months old AND the keyword has high competition (>0.71). Both risk factors present.
- `YOUNG_VOLATILE`: The page is less than 6 months old — at high risk of dropping out of the "new content" honeymoon phase.
- `HIGH_COMPETITION`: The keyword has a high competition score (>0.71) — competitors are actively trying to outrank it.

In [ ]:
import os
import pandas as pd
import numpy as np

# Load the anonymized dataset (contains content_age_days + competition)
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/raw/content_refresh_anonymized.csv'  # when running from work/notebooks/

df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} pages for baseline scoring.')
print(f'Columns used: content_age_days, competition, impressions_prev_30d')
print(f'Ground truth column: trend_direction')

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Step 1: Filter to pages with meaningful traffic
df_filtered = df[df['impressions_prev_30d'] > 100].copy()
print(f'Pages with > 100 impressions: {len(df_filtered)}')

# Step 2: Apply the two rule flags
df_filtered['is_young'] = (df_filtered['content_age_days'] < 180).astype(int)
df_filtered['is_high_comp'] = (df_filtered['competition'] > 0.71).astype(int)

# Step 3: Calculate score = max(is_young, is_high_comp) * impressions_prev_30d
df_filtered['score'] = df_filtered[['is_young', 'is_high_comp']].max(axis=1) * df_filtered['impressions_prev_30d']

# Step 4: Assign reason codes
def get_reason_code(row):
    if row['is_young'] and row['is_high_comp']:
        return 'YOUNG_AND_HIGH_COMP'
    elif row['is_young']:
        return 'YOUNG_VOLATILE'
    elif row['is_high_comp']:
        return 'HIGH_COMPETITION'
    else:
        return 'NONE'

df_filtered['reason_code'] = df_filtered.apply(get_reason_code, axis=1)

# Step 5: Rank and save
df_queue = df_filtered[df_filtered['score'] > 0].copy()
df_queue = df_queue.sort_values(by='score', ascending=False)
df_queue['rank'] = range(1, len(df_queue) + 1)

out_dir = 'work/outputs' if os.path.exists('work') else '../outputs'
os.makedirs(out_dir, exist_ok=True)
df_queue.to_csv(f'{out_dir}/baseline_action_score.csv', index=False)

print(f'Ranked queue: {len(df_queue)} pages flagged for review')
print(f'Reason code breakdown:')
print(df_queue['reason_code'].value_counts().to_string())

In [ ]:
# --- Precision@K evaluation ---

def precision_at_k(scores, labels, k):
    """Of the top-K scored items, what fraction actually declined?"""
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Ground truth: did the page actually decline?
df_filtered['label_down'] = (df_filtered['trend_direction'] == 'down').astype(int)
base_rate = df_filtered['label_down'].mean()

print(f'Base rate (random picking): {base_rate:.4f}')
print()
for k in [20, 50, 100, 500]:
    p = precision_at_k(df_filtered['score'], df_filtered['label_down'], k)
    lift = p / base_rate if base_rate > 0 else 0
    print(f'Precision@{k}: {p:.4f}  (base rate {base_rate:.4f}, lift {lift:.2f}x)')

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Show the Top-20 with the actual trend_direction for manual review
top20 = df_queue[['rank', 'content_id', 'competition', 'content_age_days',
                  'impressions_prev_30d', 'score', 'reason_code', 'trend_direction']].head(20)
display(top20)

n_correct = (top20['trend_direction'] == 'down').sum()
print(f'\nOf the top 20 flagged pages, {n_correct} actually declined ({n_correct/20:.0%}).')

### Top-20 review notes

- **Ranks 1-7, 9-16, 18-20 (YOUNG_VOLATILE)**:
  - **Action**: Review for content decay — these young, high-traffic pages risk dropping out of the "new content" honeymoon phase.
  - **Confidence**: Moderate. Young pages are directionally more volatile, but many of them may be perfectly stable evergreen topics.
  - **What would make it wrong**: If the page covers an evergreen topic with no seasonal component, being "young" alone may not indicate risk.

- **Rank 8 (HIGH_COMPETITION, competition = 1.0)**:
  - **Action**: Backlink/authority review — this older page (280 days) faces maximum keyword competition.
  - **Confidence**: High. Maximum competition score is a strong signal.
  - **What would make it wrong**: The page might have dominant domain authority that insulates it from competitor pressure.

- **Rank 17 (YOUNG_AND_HIGH_COMP)**:
  - **Action**: Immediate content refresh — both risk factors present simultaneously.
  - **Confidence**: Very High. Young (111 days) + high competition (0.86).
  - **What would make it wrong**: The keyword might have very stable search intent where the current page perfectly satisfies users.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks analysis: which top-20 pages did NOT actually decline?
weak = top20[top20['trend_direction'] != 'down']
print(f'Weak picks ({len(weak)} of 20 did not decline):')
display(weak[['rank', 'content_id', 'reason_code', 'trend_direction']])

print('\n--- Leakage check ---')
print('Columns used in score construction:')
print('  - content_age_days     (static content property, known before prediction)')
print('  - competition          (static keyword metadata, known before prediction)')
print('  - impressions_prev_30d (historical traffic from days 31-60, known before prediction)')
print()
print('Columns NOT used (would be leakage):')
print('  - trend_direction      (the label itself)')
print('  - trend_pct            (derived from the label)')
print('  - impressions_last_30d (overlaps with the prediction window)')
print()
print('Result: PASSED — no future data or product flags used in score construction.')

### Weak picks summary

- The rule heavily favors `YOUNG_VOLATILE` over `HIGH_COMPETITION` because young pages happen to have higher raw traffic in our dataset. This floods the queue with young pages and may dilute attention from older, high-competition pages that are genuinely at risk.
- Several top picks are actually "stable" or "up" — the rule is overly pessimistic about young pages that are doing fine. This is a known weakness of our simple baseline.

### Leakage check result

- We strictly used `content_age_days`, `competition`, and `impressions_prev_30d` — all available before prediction time.
- No future performance data was used. **Zero leakage confirmed.**

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.